# ANRF AISEHack 2.0 — Trees + polyBERT blend

Tests whether **polyBERT embeddings add decorrelated signal to the tree ensemble** — the only way
polyBERT can help, since alone it scored 0.862 CV vs the trees' 0.911.

**Pipeline (per target):**
- Fingerprints (RDKit desc + ECFP4/6 + RDKitFP + MACCS) → **LGBM + XGB**
- polyBERT 600-dim embeddings → **Ridge + XGB-emb**
- All four models share the **same KFold splits**, so OOF aligns → **honest nested blend** of all 4.

Prints the full error-correlation matrix (watch polyBERT-vs-tree) and each model's blend weight.
If polyBERT earns real weight and lifts honest R², submit and check the LB.

**Fast config** (2 seeds × 5 folds, 800 estimators) for a ~30-min read. Enable **GPU** + **Internet**.


In [1]:
!pip install -q rdkit sentence-transformers

import glob, os, gc, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb

# Force anonymous HF access (Kaggle's implicit token can 401 on public repos).
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
for _k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN"):
    os.environ.pop(_k, None)

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator
import torch
from sentence_transformers import SentenceTransformer

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_GPU = torch.cuda.is_available()
print("device:", DEVICE)

SEEDS = [42, 7]
N_FOLDS = 5
N_ESTIMATORS = 800
POLYBERT_PATH = "HAYDERphd/polyBERT"
TARGETS = ("tg", "egc")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 57.2 MB/s eta 0:00:00
device: cuda


In [2]:
# ---- load data ----
print("Contents of /kaggle/input:")
for root, _, files in os.walk("/kaggle/input"):
    for fn in files:
        print("  ", os.path.join(root, fn))
train_hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
test_hits  = glob.glob("/kaggle/input/**/test.csv",  recursive=True)
assert train_hits and test_hits, "train.csv/test.csv not found — add the competition dataset as input."
train = pd.read_csv(train_hits[0]); test = pd.read_csv(test_hits[0])

def split_by_type(name):
    tr = train[train["target_type"] == name].reset_index(drop=True)
    te = test[test["target_type"] == name].reset_index(drop=True)
    return tr, te, tr["target"].values.astype(float)

Contents of /kaggle/input:
   /kaggle/input/competitions/aisehack-2-0/sample_submission.csv
   /kaggle/input/competitions/aisehack-2-0/base_line_model.ipynb
   /kaggle/input/competitions/aisehack-2-0/train.csv
   /kaggle/input/competitions/aisehack-2-0/test.csv


In [3]:
# ---- fingerprint features ----
DESC_NAMES  = [n for n, _ in Descriptors.descList]
G_ECFP4 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
G_ECFP6 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
G_RDK   = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)

def featurize(smiles_list):
    d, e4, e6, rk, mc = [], [], [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            d.append([np.nan]*len(DESC_NAMES)); e4.append(np.zeros(2048,np.uint8))
            e6.append(np.zeros(2048,np.uint8)); rk.append(np.zeros(2048,np.uint8)); mc.append(np.zeros(167,np.uint8))
        else:
            d.append(list(Descriptors.CalcMolDescriptors(mol).values()))
            e4.append(G_ECFP4.GetFingerprintAsNumPy(mol)); e6.append(G_ECFP6.GetFingerprintAsNumPy(mol))
            rk.append(G_RDK.GetFingerprintAsNumPy(mol)); mc.append(np.array(MACCSkeys.GenMACCSKeys(mol),np.uint8))
    return pd.concat([pd.DataFrame(d, columns=DESC_NAMES),
                      pd.DataFrame(e4, columns=[f"e4_{i}" for i in range(2048)]),
                      pd.DataFrame(e6, columns=[f"e6_{i}" for i in range(2048)]),
                      pd.DataFrame(rk, columns=[f"rk_{i}" for i in range(2048)]),
                      pd.DataFrame(mc, columns=[f"mc_{i}" for i in range(167)])], axis=1)

def build_prep(Xr):
    X = Xr.replace([np.inf,-np.inf], np.nan).astype(float)
    X = X.dropna(axis=1, thresh=int(0.2*len(X))); X = X.loc[:, X.var() > 0]
    good = X.columns.tolist(); imp = SimpleImputer(strategy="median"); sc = StandardScaler()
    return sc.fit_transform(imp.fit_transform(X)), (imp, sc, good)

def apply_prep(Xr, prep):
    imp, sc, good = prep
    X = Xr.replace([np.inf,-np.inf], np.nan).astype(float).reindex(columns=good, fill_value=np.nan)
    return sc.transform(imp.transform(X))

In [4]:
# ---- trees (GPU-accelerated) ----
def _lgbm_gpu_ok():
    if not USE_GPU: return False
    try:
        lgb.LGBMRegressor(device="gpu", n_estimators=5, verbose=-1).fit(np.random.rand(64,6), np.random.rand(64)); return True
    except Exception as e:
        print("  LightGBM GPU unavailable -> CPU:", str(e)[:80]); return False
LGBM_GPU = _lgbm_gpu_ok()
print(f"Trees: XGBoost on {'cuda' if USE_GPU else 'cpu'}, LightGBM on {'gpu' if LGBM_GPU else 'cpu'}")

def lgbm_params(t):
    p = dict(objective="regression", metric="rmse", n_estimators=N_ESTIMATORS, learning_rate=0.01,
             num_leaves=127, max_depth=-1, min_child_samples=15, subsample=0.8, subsample_freq=1,
             colsample_bytree=0.4, reg_alpha=0.05, reg_lambda=1.0, n_jobs=-1, verbose=-1)
    if LGBM_GPU: p["device"] = "gpu"
    if t == "egc": p["num_leaves"], p["min_child_samples"] = 63, 20
    return p

def xgb_params(t):
    p = dict(objective="reg:squarederror", n_estimators=N_ESTIMATORS, learning_rate=0.01, max_depth=6,
             min_child_weight=5, subsample=0.8, colsample_bytree=0.4, reg_alpha=0.05, reg_lambda=1.0,
             n_jobs=-1, tree_method="hist", device=("cuda" if USE_GPU else "cpu"), early_stopping_rounds=200)
    if t == "egc": p["max_depth"] = 5
    return p

def train_trees(Xtr, y, Xte, t, seeds, n_splits):
    Xtr, y, Xte = np.asarray(Xtr, np.float32), np.asarray(y, float), np.asarray(Xte, np.float32)
    ol, ox, tl, tx = (np.zeros(len(Xtr)), np.zeros(len(Xtr)), np.zeros(len(Xte)), np.zeros(len(Xte)))
    for seed in seeds:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)   # SAME splits as heads
        lp = lgbm_params(t); lp["random_state"] = seed
        xp = xgb_params(t);  xp["random_state"] = seed
        for tr, va in kf.split(Xtr):
            ml = lgb.LGBMRegressor(**lp)
            ml.fit(Xtr[tr], y[tr], eval_set=[(Xtr[va], y[va])],
                   callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
            ol[va] += ml.predict(Xtr[va]) / len(seeds); tl += ml.predict(Xte) / (len(seeds)*n_splits)
            mx = xgb.XGBRegressor(**xp); mx.fit(Xtr[tr], y[tr], eval_set=[(Xtr[va], y[va])], verbose=False)
            ox[va] += mx.predict(Xtr[va]) / len(seeds); tx += mx.predict(Xte) / (len(seeds)*n_splits)
    return {"lgbm": ol, "xgb": ox}, {"lgbm": tl, "xgb": tx}

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Trees: XGBoost on cuda, LightGBM on gpu


In [5]:
# ---- polyBERT heads ----
def to_psmiles(smi): return smi.replace("[*]", "*").replace("*", "[*]")
print("Loading polyBERT ...")
polybert = SentenceTransformer(POLYBERT_PATH, device=DEVICE, token=False)
def embed(smiles_list):
    return polybert.encode([to_psmiles(s) for s in smiles_list], batch_size=256,
                           convert_to_numpy=True, show_progress_bar=False).astype(np.float32)

def train_heads(Xtr, y, Xte, seeds, n_splits):
    Xtr, y, Xte = np.asarray(Xtr, np.float32), np.asarray(y, float), np.asarray(Xte, np.float32)
    orr, oxx, terr, texx = (np.zeros(len(Xtr)), np.zeros(len(Xtr)), np.zeros(len(Xte)), np.zeros(len(Xte)))
    for seed in seeds:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)   # SAME splits as trees
        for tr, va in kf.split(Xtr):
            sc = StandardScaler().fit(Xtr[tr])
            rg = RidgeCV(alphas=np.logspace(-2,4,25)).fit(sc.transform(Xtr[tr]), y[tr])
            orr[va] += rg.predict(sc.transform(Xtr[va]))/len(seeds); terr += rg.predict(sc.transform(Xte))/(len(seeds)*n_splits)
            mx = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=2000, learning_rate=0.02,
                                  max_depth=4, min_child_weight=5, subsample=0.8, colsample_bytree=0.6,
                                  reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, tree_method="hist",
                                  device=("cuda" if USE_GPU else "cpu"), early_stopping_rounds=100, random_state=seed)
            mx.fit(Xtr[tr], y[tr], eval_set=[(Xtr[va], y[va])], verbose=False)
            oxx[va] += mx.predict(Xtr[va])/len(seeds); texx += mx.predict(Xte)/(len(seeds)*n_splits)
    return {"ridge": orr, "xgb_emb": oxx}, {"ridge": terr, "xgb_emb": texx}

Loading polyBERT ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/84.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# ---- honest nested blend ----
def _fit_w(P, y):
    n = P.shape[1]
    r = minimize(lambda w: -r2_score(y, P @ w), np.full(n,1/n), method="SLSQP",
                 bounds=[(0,1)]*n, constraints={"type":"eq","fun":lambda w: w.sum()-1})
    return r.x
def honest_cv_r2(oof, y, n_folds=5, seed=SEED):
    P = np.column_stack([oof[m] for m in oof]); y = np.asarray(y, float)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed); pred = np.zeros(len(y))
    for tr, va in kf.split(P): pred[va] = P[va] @ _fit_w(P[tr], y[tr])
    return r2_score(y, pred)
def final_weights(oof, y):
    return dict(zip(oof, _fit_w(np.column_stack([oof[m] for m in oof]), np.asarray(y, float))))
def error_corr(oof, y):
    y = np.asarray(y, float); e = {m: oof[m]-y for m in oof}; names = list(oof); out = {}
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            out[f"{names[i]}~{names[j]}"] = round(float(np.corrcoef(e[names[i]], e[names[j]])[0,1]), 3)
    return out

In [7]:
# ---- run: trees + polyBERT -> honest 4-way blend -> submission ----
parts, honest_by_target, tree_only = [], {}, {}
for name in TARGETS:
    print("="*60, f"\n  {name.upper()}\n", "="*60)
    tr, te, y = split_by_type(name)

    Xtr, prep = build_prep(featurize(tr["smiles"].tolist()))
    Xte = apply_prep(featurize(te["smiles"].tolist()), prep)
    oof_t, testp_t = train_trees(Xtr, y, Xte, name, SEEDS, N_FOLDS)

    Etr, Ete = embed(tr["smiles"].tolist()), embed(te["smiles"].tolist())
    oof_h, testp_h = train_heads(Etr, y, Ete, SEEDS, N_FOLDS)

    oof   = {**oof_t, **oof_h}
    testp = {**testp_t, **testp_h}

    per = {m: round(r2_score(y, oof[m]),4) for m in oof}
    w   = final_weights(oof, y)
    h   = honest_cv_r2(oof, y)
    h_trees = honest_cv_r2(oof_t, y)   # trees-only, same helper, for comparison
    blend = sum(w[m]*testp[m] for m in w)

    print("  per-model OOF R²:", per)
    print("  error corr:", error_corr(oof, y))
    print("  weights:", {m: round(v,3) for m,v in w.items()})
    print(f"  HONEST R²: trees-only={h_trees:.4f}  ->  trees+polyBERT={h:.4f}  (delta {h-h_trees:+.4f})")
    honest_by_target[name] = h; tree_only[name] = h_trees
    sub = te[["id"]].copy(); sub["target"] = blend; parts.append(sub)

submission = pd.concat(parts, axis=0).sort_values("id").reset_index(drop=True)
assert submission["target"].isna().sum() == 0
submission.to_csv("submission.csv", index=False)

mh, mt = np.mean(list(honest_by_target.values())), np.mean(list(tree_only.values()))
print("\n" + "="*60)
print(f"  MEAN HONEST R²: trees-only={mt:.4f}  ->  trees+polyBERT={mh:.4f}  (delta {mh-mt:+.4f})")
print("  (fingerprint+tree full-run baseline: 0.9114 CV / 0.895 LB)")
print("  submission rows:", len(submission))
print("="*60)
print(submission.head().to_string())

  TG


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  per-model OOF R²: {'lgbm': 0.8979, 'xgb': 0.8898, 'ridge': 0.8331, 'xgb_emb': 0.8626}
  error corr: {'lgbm~xgb': 0.978, 'lgbm~ridge': 0.723, 'lgbm~xgb_emb': 0.821, 'xgb~ridge': 0.748, 'xgb~xgb_emb': 0.838, 'ridge~xgb_emb': 0.801}
  weights: {'lgbm': np.float64(0.871), 'xgb': np.float64(0.0), 'ridge': np.float64(0.075), 'xgb_emb': np.float64(0.054)}
  HONEST R²: trees-only=0.8979  ->  trees+polyBERT=0.8986  (delta +0.0007)
  EGC
  per-model OOF R²: {'lgbm': 0.9007, 'xgb': 0.9024, 'ridge': 0.8352, 'xgb_emb': 0.8503}
  error corr: {'lgbm~xgb': 0.957, 'lgbm~ridge': 0.695, 'lgbm~xgb_emb': 0.773, 'xgb~ridge': 0.71, 'xgb~xgb_emb': 0.79, 'ridge~xgb_emb': 0.818}
  weights: {'lgbm': np.float64(0.449), 'xgb': np.float64(0.466), 'ridge': np.float64(0.085), 'xgb_emb': np.float64(0.0)}
  HONEST R²: trees-only=0.9031  ->  trees+polyBERT=0.9037  (delta +0.0006)

  MEAN HONEST R²: trees-only=0.9005  ->  trees+polyBERT=0.9011  (delta +0.0006)
  (fingerprint+tree full-run baseline: 0.9114 CV / 0.895 LB